In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import logging
import os
from IPython.display import display

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.results_analyser.prompt_effect_analyser import PromptEffectAnalyser
from gsm_benchmarker.results_analyser.plotting_utils import plot_glmm, plot_acc_change_distribution, Colour, plot_prompt_format_comparison

logger = logging.getLogger('notebook')

plt.style.use('default')
plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
def display_glmm(*args, **kwargs):
    fig_or, fig_bars = plot_glmm(*args, **kwargs)
    display(fig_or)
    display(fig_bars)
    return fig_or, fig_bars


In [ ]:
METRIC = "correct"

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()

p_standard_pilot = pp / "mini_20x50x4__14_11/final"
p_standard = pp / "default_full__06_02/final"

p_code = pp / 'code_full__09_02/corrected'
p_formal = pp / 'formalised_full__16_04/final'
p_nonformal = pp / 'nonformalised_full__20_04/final'

In [ ]:
mres_standard_pilot = MultiVariantMultiModelResultsAnalyser(p_standard_pilot)

mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)

mres_code = MultiVariantMultiModelResultsAnalyser(p_code)
mres_formal = MultiVariantMultiModelResultsAnalyser(p_formal)
mres_nonformal = MultiVariantMultiModelResultsAnalyser(p_nonformal)


In [ ]:
pea_code = PromptEffectAnalyser(mres_standard, mres_code, "Code output prompt")
pea_formal = PromptEffectAnalyser(mres_standard, mres_formal, "Formalised NL prompt")
pea_nonformal = PromptEffectAnalyser(mres_standard, mres_nonformal, "Non-formalised NL prompt")

In [ ]:
COLOURS = {
    'standard': Colour('green'),
    'code': Colour('rebeccapurple'),
    'formal': Colour('steelblue'),
    'nonformal': Colour('teal')
}

In [ ]:
OUTPUTS = Path("./outputs").resolve()
os.makedirs(OUTPUTS, exist_ok=True)
OUTPUTS_FOLDER = str(OUTPUTS) + "/"

### Modelling question difficulty

In [ ]:
pea_code._baseline_mres.plot_question_difficulty_per_model(
    # title="Leave-one-(model-)out question difficulty",
    save_prefix=OUTPUTS_FOLDER,
)

In [ ]:
fig_qd_hist = pea_code._baseline_mres.plot_question_difficulty_histogram(
    n_levels=5,
    color=COLOURS['standard'].value,
    cumulative_color=COLOURS['standard'].darken(factor=0.6),
    save_prefix=OUTPUTS_FOLDER,
)
display(fig_qd_hist)

In [ ]:
mres_standard.plot_number_counts(save_prefix=OUTPUTS_FOLDER)

## Question 1
Are the accuracy drops reported in the GSM-Symbolic paper actually significant?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with GSM-Symbolic prompt.

### 1A: Pilot run
Checking on a subset of the benchmark first. Projecting results to the full dataset with alpha = 20%.

50/100 questions, 20/50 template variations -> 1000/5000 total questions in the 'main variant' (20% of the benchmark) (+ 50 questions in the original GSM8K variant)

In [ ]:
ALPHA = 0.05
ALPHA_THRESHOLD = 1.1 * ALPHA
PROJECTED_ALPHA = 0.2
PROJECTED_ALPHA_THRESHOLD = 1.1 * PROJECTED_ALPHA

In [ ]:
variant_effect_df_pilot = mres_standard_pilot.analyse_variant_effect('main', metric=METRIC)

In [ ]:
display_glmm(
    variant_effect_df_pilot,
    'accuracy_change',
    "Symbolic performance delta",
    # title="Accuracy change due to templates (pilot run)",
    projected_alpha=PROJECTED_ALPHA,
    bar_colour=COLOURS['standard'].value,
    save_prefix=OUTPUTS/"standard_pilot"
)


#### Candidate models
Models which, based on the pilot results, are worth checking on the full dataset.

In [ ]:
candidate_models = variant_effect_df_pilot[variant_effect_df_pilot.p_value < PROJECTED_ALPHA_THRESHOLD].index.get_level_values('model').unique().tolist()
candidate_models

### 1B: Full benchmark
Re-running the tests on the full benchmark (100 + 5000 questions) with the identified candidate models.

In [ ]:
variant_effect_df_full = mres_standard.analyse_variant_effect('main', models=candidate_models, metric=METRIC)
variant_effect_df_full

In [ ]:
display_glmm(
    variant_effect_df_full,
    'accuracy_change',
    "Symbolic performance delta",
    # title="Accuracy change due to templates (full benchmark)",
    bar_colour=COLOURS['standard'].value,
    save_prefix=OUTPUTS/"standard_full"
)

In [ ]:
significant_models = variant_effect_df_full[variant_effect_df_full.p_value < ALPHA_THRESHOLD].index.get_level_values('model').unique().tolist()
significant_models

## Question 2
Do alternative prompt formats remove the variant dependency?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with code prompt and formalised natural language prompt.

In [ ]:
variant_effect_df_code = mres_code.analyse_variant_effect('main', models=significant_models, metric=METRIC)
display_glmm(
    variant_effect_df_code,
    'accuracy_change',
    "Symbolic performance delta",
    # title="Accuracy change due to templates - code output prompt",
    bar_colour=COLOURS['code'].value,
    save_prefix=OUTPUTS/"code"
)

In [ ]:
variant_effect_df_formal = mres_formal.analyse_variant_effect('main', models=significant_models, metric=METRIC)
display_glmm(
    variant_effect_df_formal,
    'accuracy_change',
    "Symbolic performance delta",
    # title="Accuracy change due to templates - formalised NL prompt",
    bar_colour=COLOURS['formal'].value,
    save_prefix=OUTPUTS/"formal"
)

In [ ]:
variant_effect_df_nonformal = mres_nonformal.analyse_variant_effect('main', models=significant_models, metric=METRIC)
display_glmm(
    variant_effect_df_nonformal,
    'accuracy_change',
    "Symbolic performance delta",
    # title="Accuracy change due to templates - non-formalised NL prompt",
    bar_colour=COLOURS['nonformal'].value,
    save_prefix=OUTPUTS/"nonformal"
)

### Alternative prompts evaluation
Do alternative prompt formats result in significant performance changes?

Evaluating performance on 'main' variant with the alternative prompts vs GSM-Symbolic prompt.

In [ ]:
acc_change_df_code = pea_code.analyse_accuracy_change_significance(variant='main', models=significant_models, metric=METRIC)
acc_change_df_code

In [ ]:
display_glmm(
    acc_change_df_code,
    'mean_diff',
    "Prompt performance delta",
    # title="Effect of code prompt on accuracy on 'main' variant",
    bar_colour=COLOURS['code'].lighten(),
    save_prefix=OUTPUTS/"code_pe"
)

In [ ]:
acc_change_raw = pea_code.get_raw_acc_change(variant='main', metric=METRIC)
plot_acc_change_distribution(
    acc_change_raw,
    label="Prompt performance delta",
    models=significant_models,
    color=COLOURS['code'].lighten(),
    save_prefix=OUTPUTS/"code_pe"
)

In [ ]:
acc_change_df_formal = pea_formal.analyse_accuracy_change_significance(variant='main', models=significant_models, metric=METRIC)
display_glmm(
    acc_change_df_formal,
    'mean_diff',
    "Prompt performance delta",
    # title="Effect of formalised NL prompt on accuracy on 'main' variant",
    bar_colour=COLOURS['formal'].lighten(),
    save_prefix=OUTPUTS/"formal_pe"
)

In [ ]:
acc_change_df_nonformal = pea_nonformal.analyse_accuracy_change_significance(variant='main', models=significant_models, metric=METRIC)
display_glmm(
    acc_change_df_nonformal,
    'mean_diff',
    "Prompt performance delta",
    # title="Effect of non-formalised NL prompt on accuracy on 'main' variant",
    bar_colour=COLOURS['nonformal'].lighten(),
    save_prefix=OUTPUTS/"nonformal_pe"
)

### Mean accuracy by prompt format on `main` (significant models)

In [ ]:
prompt_order = ['standard', 'nonformal', 'formal', 'code']
prompt_labels = {
    'standard': 'GSM prompt',
    'code': 'Code prompt',
    'formal': 'Formalised NL prompt',
    'nonformal': 'Non-formalised NL prompt',
}
prompt_colours = {key: COLOURS[key].value for key in prompt_order}
prompt_mres = {
    'standard': mres_standard,
    'code': mres_code,
    'formal': mres_formal,
    'nonformal': mres_nonformal,
}

acc_records = []
prompt_effect_records = []
variant_records = []

existing_variant_dfs = {
    'standard': variant_effect_df_full,
    'code': variant_effect_df_code,
    'formal': variant_effect_df_formal,
    'nonformal': variant_effect_df_nonformal,
}

existing_prompt_effect_dfs = {
    'code': acc_change_df_code,
    'formal': acc_change_df_formal,
    'nonformal': acc_change_df_nonformal,
}

for prompt_key in prompt_order:
    mres = prompt_mres[prompt_key]

    acc_per_model = (
        mres
        .variants['main']
        .get_accuracies_per_model_and_template_id(metric=METRIC)
        .groupby('model')
        .mean()
    )

    for model in significant_models:
        if model in acc_per_model.index:
            acc_records.append({
                'model': model,
                'prompt': prompt_key,
                'mean_accuracy': float(acc_per_model.loc[model]),
            })

    variant_df_prompt = existing_variant_dfs[prompt_key]
    for model in significant_models:
        if model in variant_df_prompt.index:
            variant_records.append({
                'model': model,
                'prompt': prompt_key,
                'accuracy_change': float(variant_df_prompt.loc[model, 'accuracy_change']),
                'p_value': float(variant_df_prompt.loc[model, 'p_value']),
            })

    if prompt_key == 'standard':
        for model in significant_models:
            prompt_effect_records.append({
                'model': model,
                'prompt': prompt_key,
                'prompt_accuracy_change': float('nan'),
                'prompt_p_value': float('nan'),
            })
    else:
        prompt_effect_df_prompt = existing_prompt_effect_dfs[prompt_key]
        for model in significant_models:
            if model in prompt_effect_df_prompt.index:
                prompt_effect_records.append({
                    'model': model,
                    'prompt': prompt_key,
                    'prompt_accuracy_change': float(prompt_effect_df_prompt.loc[model, 'mean_diff']),
                    'prompt_p_value': float(prompt_effect_df_prompt.loc[model, 'p_value']),
                })

acc_df = pd.DataFrame(acc_records)
prompt_effect_df = pd.DataFrame(prompt_effect_records)
variant_df = pd.DataFrame(variant_records)
plot_df = (
    acc_df
    .merge(prompt_effect_df, on=['model', 'prompt'], how='left')
    .merge(variant_df, on=['model', 'prompt'], how='left')
)

fig = plot_prompt_format_comparison(
    plot_df,
    selected_models=significant_models,
    prompt_order=prompt_order,
    prompt_labels=prompt_labels,
    prompt_colours=prompt_colours,
    mean_ylabel='Mean accuracy',
    prompt_effect_ylabel='Prompt performance delta',
    variant_ylabel='Symbolic performance delta',
    save_prefix=OUTPUTS_FOLDER,
)
display(fig)
